# 01 — EDA: TrashNet (dataset-resized)

Eksploracja lokalnego zbioru po `scripts/download_data.py` i `scripts/prepare_split.py`.

**Cel:** liczebności klas, rozdzielczości, uszkodzone pliki, podział train/val/test, przykładowe obrazy i krótkie wnioski pod trening.

In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from sortwise.config import CLASS_LABELS_PL, CLASS_NAMES, DATASET_DIR, MANIFEST_PATH, SPLITS_DIR

print("ROOT:", ROOT)
print("DATASET:", DATASET_DIR)
print("exists:", DATASET_DIR.is_dir())

In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
files = pd.DataFrame(manifest["files"])
print("num_files:", manifest["num_files"])
print("corrupt:", len(manifest.get("corrupt_files", [])))
display(files.head())

## Liczebność klas

In [ ]:
counts = files["class"].value_counts().reindex(CLASS_NAMES)
counts_pl = counts.rename(index=CLASS_LABELS_PL)
display(counts_pl.to_frame("n"))

fig, ax = plt.subplots(figsize=(8, 4))
counts.plot(kind="bar", ax=ax, color="#2f6f4e")
ax.set_title("TrashNet — liczba zdjęć per klasa")
ax.set_xlabel("klasa")
ax.set_ylabel("liczba zdjęć")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

print(f"Imbalance ratio (max/min): {counts.max() / counts.min():.2f}")
print("Uwaga: klasa `trash` jest wyraźnie mniejsza — macro-F1 będzie ważniejsze niż accuracy.")

## Rozdzielczości i poprawność plików

In [ ]:
ok = files[files["ok"] == True].copy()
print("usable:", len(ok), "/ corrupt:", (~files["ok"]).sum())

res = ok.groupby(["width", "height"]).size().sort_values(ascending=False)
display(res.head(10).to_frame("n"))

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(ok["width"], ok["height"], alpha=0.25, s=12)
ax.set_title("Rozdzielczości zdjęć")
ax.set_xlabel("width")
ax.set_ylabel("height")
plt.tight_layout()
plt.show()

## Split train / val / test

In [ ]:
splits_path = SPLITS_DIR / "splits.json"
splits_doc = json.loads(splits_path.read_text(encoding="utf-8"))
print("seed:", splits_doc["seed"])
print("ratios:", splits_doc["ratios"])
print("counts:", splits_doc["counts"])

rows = []
for split_name, class_counts in splits_doc["class_counts"].items():
    for cls, n in class_counts.items():
        rows.append({"split": split_name, "class": cls, "n": n})
split_df = pd.DataFrame(rows)
pivot = split_df.pivot(index="class", columns="split", values="n").reindex(CLASS_NAMES)
pivot = pivot[["train", "val", "test"]]
display(pivot)

pivot.plot(kind="bar", figsize=(9, 4))
plt.title("Rozkład klas w splitach (stratified)")
plt.ylabel("n")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Przykładowe obrazy

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
axes = axes.ravel()
for ax, cls in zip(axes, CLASS_NAMES):
    sample = files[files["class"] == cls].iloc[0]
    img = Image.open(DATASET_DIR / sample["path"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(f"{cls} / {CLASS_LABELS_PL[cls]}")
    ax.axis("off")
plt.suptitle("Przykłady (jedno zdjęcie na klasę)")
plt.tight_layout()
plt.show()

## Wnioski

1. Zbiór ma **2527** zdjęć w **6** klasach; pliki zdekodowały się poprawnie (0 corrupt w manifeście).
2. Klasa **`trash`** jest wyraźnie mniejsza — przy ewaluacji raportować **macro-F1** i metryki per klasa, nie tylko accuracy.
3. Zdjęcia pochodzą głównie z kontrolowanych warunków (jasne / białe tło) — model może gorzej generalizować na własne zdjęcia z innymi tłami; warto przygotować osobny zestaw testowy.
4. Split **70/15/15** ze seedem **42** jest zapisany w `data/processed/splits/` i powinien być używany we wszystkich eksperymentach (bez przecieku).
5. Następny krok: baseline **ResNet18** (head-only vs fine-tune) na CUDA i porównanie z drugim modelem.